In [94]:
import os
from sedona.spark import SedonaContext
import geopandas as gpd
from time import time
import pyspark.sql.functions as f
from sedona.spark import dataframe_to_arrow
import geopandas as gpd
from sedona.utils.geoarrow import create_spatial_dataframe

## GeoPandas DataFrame from Sedona Spatial DataFrame

In [2]:
additional_packages = [
    'org.apache.sedona:sedona-spark-3.5_2.12:1.7.1',
    'org.datasyslab:geotools-wrapper:1.7.1-28.5',
]

config_params = {
    "spark.jars.packages": ",".join(additional_packages)   
}

if os.environ.get("SEDONA_COPY_MINIO") == "true":
    config_params = {
        **config_params,
        **{
            "spark.hadoop.fs.s3a.access.key": "sedona",
            "spark.hadoop.fs.s3a.secret.key": "sedona_password",
            "spark.hadoop.fs.s3a.endpoint": "http://minio:9000",
            "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
            "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
            "spark.hadoop.fs.s3a.path.style.access": "true"
        }
    }

bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

for key, value in config_params.items():
    config = config.config(key, value)

sedona = SedonaContext.create(config.getOrCreate())
sedona.sparkContext.setLogLevel("ERROR")

sedona.sparkContext.setCheckpointDir("checkpoint")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9f623566-df03-496c-861b-960d30c6aa52;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-3.5_2.12;1.7.1 in central
	found org.apache.sedona#sedona-common;1.7.1 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.wololo#jts2geojson;0.16.1 in central
	found org.locationtech.spatial4j#spatial4j;0.8 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-

In [42]:
df = sedona.read.format("geoparquet").\
    load(f"s3a://apache-sedona-book/source_data/transportation_barcelona/barcelona.geoparquet").\
    withColumn("geom", f.col("geometry")).\
    drop("geometry")

# Sedona DataFrame to GeoPandas

In [43]:
## Inefficient way of converting spatial DataFrame to GeoPandas

In [44]:
import geopandas as gpd

start = time()
gdf = gpd.GeoDataFrame(df.toPandas(), geometry="geom")
print(f"converted in {time() - start}")

converted in 8.842872858047485


In [45]:
## using the GeoArrow conversion

In [46]:
start = time()
gdf = gpd.GeoDataFrame.from_arrow(dataframe_to_arrow(df))
print(f"converted in {time() - start}")

converted in 1.5720555782318115


# Creating Sedona DataFrame from shapely objects

In [47]:
from shapely.geometry import Point
import sedona.sql.types as st
import pyspark.sql.types as t
 
schema = t.StructType(
    [
        t.StructField("id", t.IntegerType()),
        t.StructField("geom", st.GeometryType()),
    ]
)
 
shapely_df = sedona.createDataFrame([
    [1, Point(21, 52)],
    [2, Point(21, 45)]
], schema=schema)

In [48]:
shapely_df.show()

+---+-------------+
| id|         geom|
+---+-------------+
|  1|POINT (21 52)|
|  2|POINT (21 45)|
+---+-------------+



In [49]:
sedona.createDataFrame([
    {"id": 1, "geom": Point(21, 52)},
    {"id": 2, "geom": Point(21, 45)}
]).show()

+-------------+---+
|         geom| id|
+-------------+---+
|POINT (21 52)|  1|
|POINT (21 45)|  2|
+-------------+---+



# Sedona DataFrame from GeoPandas

In [ ]:
gdf = gdf[["id", "bbox", "version", "subtype", "class", "geom", "names"]]

In [72]:
start = time()
sedona.createDataFrame(gdf)
print(f"converted in {time() - start}")

converted in 2.2028284072875977


In [63]:
# Sedona DataFrame from geopandas using Apache Arrow

In [69]:
start = time()
create_spatial_dataframe(sedona, gdf)
print(f"converted in {time() - start}")

converted in 0.9103448390960693


# Writing own UDF function

In [75]:
import pyspark.sql.functions as f
import sedona.sql.types as st
import shapely.geometry.base as b
 
def create_buffer_distance(
    s: b.BaseGeometry,
    distance_from: float,
    distance_to: float
) -> b.BaseGeometry:
    buffer_a = s.buffer(float(distance_from))
    buffer_b = s.buffer(float(distance_to))
    return buffer_b.difference(buffer_a)
 
buffer_distanced_udf = f.udf(create_buffer_distance, st.GeometryType())
 
sedona.udf.register(
    "ST_BufferDistanceNonVectorized",
    buffer_distanced_udf
)

In [77]:
df.createOrReplaceTempView("roads")

In [80]:
sedona.sql(
"""
    SELECT 
        ST_BufferDistanceNonVectorized(geom, 0.0001, 0.0002) AS geometry
    FROM roads
    """
).show()

[Stage 38:>                                                         (0 + 1) / 1]

+--------------------+
|            geometry|
+--------------------+
|POLYGON ((3.70448...|
|POLYGON ((5.33776...|
|POLYGON ((2.18222...|
|POLYGON ((3.70527...|
|POLYGON ((3.06424...|
|POLYGON ((8.45250...|
|POLYGON ((-0.6447...|
|POLYGON ((8.91493...|
|POLYGON ((1.44936...|
|POLYGON ((2.63515...|
|POLYGON ((2.08226...|
|POLYGON ((2.08236...|
|POLYGON ((2.08817...|
|POLYGON ((2.08926...|
|POLYGON ((2.08997...|
|POLYGON ((2.09033...|
|POLYGON ((2.09064...|
|POLYGON ((2.09172...|
|POLYGON ((2.09142...|
|POLYGON ((2.09272...|
+--------------------+
only showing top 20 rows



# Writing Vectorized UDF (better performance)

In [82]:
@sedona_vectorized_udf(return_type=GeometryType())
def vectorized_symmetrical_buffer_distance_udf(
        geom: b.BaseGeometry
) -> b.BaseGeometry:
    return create_buffer_distance(geom, 0.0001, 0.0002)

NameError: name 'sedona_vectorized_udf' is not defined

In [95]:
df.select(
    vectorized_symmetrical_buffer_distance_udf(f.col("geometry"))
).show(10)

NameError: name 'vectorized_symmetrical_buffer_distance_udf' is not defined